# ⚡ MojoPQC-SCA: Quick Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hardikxk/pqc-sca/blob/main/notebooks/quick_demo.ipynb)

---

A **fast, self-contained demo** of the MojoPQC-SCA pipeline that runs in **~5 minutes** on Colab.

Uses a smaller dataset (2K traces) for quick iteration. For full experiments, use `colab_train.ipynb`.

> **GPU optional** — runs on both CPU and GPU. Select T4 for faster training.

In [ ]:
# ── Install & clone ───────────────────────────────────────────────
!pip install -q numpy scipy h5py pyyaml psutil matplotlib torch tqdm onnx onnxruntime

import os
REPO = '/content/MojoPQC-SCA'
if not os.path.exists(REPO):
    !git clone https://github.com/hardikxk/pqc-sca.git {REPO}
os.chdir(REPO)

import sys, json, time
sys.path.insert(0, '.')
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 120

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Ready | Device: {DEVICE} | PyTorch: {torch.__version__}')

---

## 🧪 All-in-One: Generate → Preprocess → Train → Attack

This single cell runs the entire pipeline inline (no shell scripts) so you can see exactly what happens at each step.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 1: Generate synthetic ML-KEM-512 side-channel traces
# ═══════════════════════════════════════════════════════════════════
from mojopqc_sca.datasets.synthetic_pqc import generate_dataset
from mojopqc_sca.utils.config import ensure_output_dirs

# Small config for quick demo
config = {
    'seed': 2026,
    'dataset': {
        'profiling_traces': 2000,
        'attack_traces': 500,
        'trace_length': 50000,   # 50K samples (smaller for speed)
        'chunk_size': 256,
        'classes': 256,
        'noise_std': 1.0,
        'leakage_amplitude': 0.12,
    },
    'preprocessing': {
        'filter_kernel_size': 11,
        'max_shift': 250,
        'target_length': 5000,
    }
}

ensure_output_dirs()
t0 = time.time()
generate_dataset('data/raw/demo.h5', config)
print(f'✅ Generated {config["dataset"]["profiling_traces"]} profiling + '
      f'{config["dataset"]["attack_traces"]} attack traces in {time.time()-t0:.1f}s')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 2: Stream-preprocess (filter → align → downsample → normalize)
# ═══════════════════════════════════════════════════════════════════
from mojopqc_sca.preprocessing.python_baseline import preprocess_pipeline

t0 = time.time()
stats = preprocess_pipeline('data/raw/demo.h5', 'data/processed/demo.h5', config)
print(f'✅ Preprocessed in {time.time()-t0:.1f}s')
print(f'   50,000 → 5,000 samples (10× compression)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# VISUALIZE: Raw vs processed traces
# ═══════════════════════════════════════════════════════════════════
import h5py

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

with h5py.File('data/raw/demo.h5', 'r') as f:
    raw = f['traces/profiling'][0]
with h5py.File('data/processed/demo.h5', 'r') as f:
    processed = f['traces/profiling'][0]

axes[0].plot(raw[::10], linewidth=0.3, color='#FF5722')
axes[0].set_title('Raw Trace (50K samples)', fontweight='bold')
axes[0].set_ylabel('Amplitude')

axes[1].plot(processed, linewidth=0.5, color='#4CAF50')
axes[1].set_title('Processed Trace (5K samples)', fontweight='bold')
axes[1].set_ylabel('Normalized amplitude')

for ax in axes: ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 3: Train CNN + MPS model (inline with live plots)
# ═══════════════════════════════════════════════════════════════════
from mojopqc_sca.models.cnn import LightweightCNN, count_parameters
from mojopqc_sca.models.cnn_mps import CNNWithMPS
from mojopqc_sca.datasets.torch_hdf5 import HDF5TraceDataset
from torch.utils.data import DataLoader
import random

# Seed everything
seed = 2026
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# Build model
model = CNNWithMPS(5000, 256, bond_dim=8).to(DEVICE)
print(f'🧠 CNN+MPS model: {count_parameters(model):,} parameters')
print(f'   Budget usage: {count_parameters(model)/200000*100:.1f}%\n')

# Data loaders
full = HDF5TraceDataset('data/processed/demo.h5', 'profiling')
indices = np.random.default_rng(seed).permutation(len(full))
split = int(len(indices) * 0.8)
train_set = HDF5TraceDataset('data/processed/demo.h5', 'profiling', indices[:split].tolist())
val_set   = HDF5TraceDataset('data/processed/demo.h5', 'profiling', indices[split:].tolist())
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=64, shuffle=False)

# Training loop
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

EPOCHS = 20
history = {'loss': [], 'train_acc': [], 'val_acc': [], 'time': []}

print(f'Training for {EPOCHS} epochs on {DEVICE.upper()}...')
print(f'{"Epoch":>6s} {"Loss":>10s} {"Train Acc":>10s} {"Val Acc":>10s} {"Time":>8s}')
print('-' * 48)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    
    # Train
    model.train()
    total_loss = correct = count = 0
    for traces, labels in train_loader:
        traces, labels = traces.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(traces)
        loss = loss_fn(logits, labels)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        count += labels.size(0)
    
    # Validate
    model.eval()
    val_correct = val_count = 0
    with torch.no_grad():
        for traces, labels in val_loader:
            traces, labels = traces.to(DEVICE), labels.to(DEVICE)
            logits = model(traces)
            val_correct += (logits.argmax(1) == labels).sum().item()
            val_count += labels.size(0)
    
    epoch_time = time.time() - t0
    history['loss'].append(total_loss / count)
    history['train_acc'].append(correct / count)
    history['val_acc'].append(val_correct / val_count)
    history['time'].append(epoch_time)
    
    print(f'{epoch:>6d} {total_loss/count:>10.4f} {correct/count*100:>9.2f}% '
          f'{val_correct/val_count*100:>9.2f}% {epoch_time:>6.2f}s')

total_time = sum(history['time'])
print(f'\n✅ Training complete in {total_time:.1f}s')
print(f'   Final val accuracy: {history["val_acc"][-1]*100:.2f}%')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# VISUALIZE: Training curves
# ═══════════════════════════════════════════════════════════════════
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, EPOCHS + 1)
ax1.plot(epochs, history['loss'], 'o-', color='#E91E63', linewidth=2, markersize=4)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss', fontweight='bold')
ax1.grid(alpha=0.2)

ax2.plot(epochs, [a*100 for a in history['train_acc']], '--', color='#2196F3',
         linewidth=2, label='Train', alpha=0.6)
ax2.plot(epochs, [a*100 for a in history['val_acc']], 'o-', color='#2196F3',
         linewidth=2, markersize=4, label='Validation')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy', fontweight='bold')
ax2.legend(); ax2.grid(alpha=0.2)

plt.suptitle(f'CNN+MPS Training ({DEVICE.upper()})', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 4: Side-channel attack — compute guessing entropy
# ═══════════════════════════════════════════════════════════════════
from mojopqc_sca.evaluation.guessing_entropy import compute_ge_curve

# Load attack traces
attack_set = HDF5TraceDataset('data/processed/demo.h5', 'attack')
attack_loader = DataLoader(attack_set, batch_size=64, shuffle=False)

# Collect predictions
model.eval()
all_log_probs, all_labels = [], []
with torch.no_grad():
    for traces, labels in attack_loader:
        traces = traces.to(DEVICE)
        log_probs = torch.log_softmax(model(traces), dim=1).cpu().numpy()
        all_log_probs.append(log_probs)
        all_labels.append(labels.numpy())

log_probs = np.concatenate(all_log_probs)
labels    = np.concatenate(all_labels)

# Compute GE curve
ge_curve = compute_ge_curve(log_probs, labels, step=25)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
traces_n = list(ge_curve.keys())
ge_vals  = list(ge_curve.values())

ax.plot(traces_n, ge_vals, 'o-', color='#E91E63', linewidth=2,
        markersize=5, markerfacecolor='white', markeredgewidth=1.5)
ax.axhline(y=1, color='#4CAF50', linestyle='--', linewidth=2,
           label='Full key recovery (GE=1)')
ax.fill_between(traces_n, ge_vals, alpha=0.1, color='#E91E63')
ax.set_xlabel('Number of Attack Traces', fontsize=12)
ax.set_ylabel('Guessing Entropy (key rank)', fontsize=12)
ax.set_title('🔑 Key Recovery — Guessing Entropy Curve', fontweight='bold', fontsize=14)
ax.set_yscale('log')
ax.legend(fontsize=11)
ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

print(f'\n📊 Attack Results:')
print(f'   GE at {traces_n[0]} traces : {ge_vals[0]:.1f}')
print(f'   GE at {traces_n[-1]} traces: {ge_vals[-1]:.1f}')
for n, ge in zip(traces_n, ge_vals):
    if ge <= 1:
        print(f'   🔑 KEY RECOVERED at {n} traces!')
        break

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BONUS: MPS Tensor Core Visualization
# ═══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle('MPS Tensor Cores (bond_dim=8, 5 sites)', fontweight='bold', fontsize=13)

for i, core in enumerate(model.mps.cores):
    # Each core is (bond_dim, 2, bond_dim) = (8, 2, 8)
    # Visualize the two slices
    w = core.detach().cpu().numpy()
    ax = axes[i]
    # Show mean of the two physical indices
    im = ax.imshow(w.mean(axis=1), cmap='RdBu_r', aspect='auto',
                   vmin=-0.1, vmax=0.1)
    ax.set_title(f'Core {i+1}', fontsize=10)
    ax.set_xlabel('Right bond')
    if i == 0: ax.set_ylabel('Left bond')
    ax.set_xticks(range(0, 8, 2))
    ax.set_yticks(range(0, 8, 2))

plt.colorbar(im, ax=axes, label='Weight magnitude', shrink=0.8)
plt.tight_layout()
plt.show()

# Norm analysis
print('\n🔍 Core Frobenius norms (smaller = more regularized):')
for i, core in enumerate(model.mps.cores):
    norm = torch.norm(core).item()
    print(f'   Core {i+1}: {norm:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════
print('\n' + '=' * 55)
print('🎯 DEMO SUMMARY')
print('=' * 55)
print(f'Model           : CNN + MPS (bond_dim=8)')
print(f'Parameters      : {count_parameters(model):,}')
print(f'Training epochs : {EPOCHS}')
print(f'Training time   : {total_time:.1f}s ({DEVICE.upper()})')
print(f'Val accuracy    : {history["val_acc"][-1]*100:.2f}%')
print(f'Final GE        : {ge_vals[-1]:.1f} (at {traces_n[-1]} traces)')
print(f'Device          : {DEVICE.upper()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'Peak GPU mem    : {torch.cuda.max_memory_allocated()/1e6:.1f} MB')
print('=' * 55)
print('\n🔗 For full experiments, use: notebooks/colab_train.ipynb')
print('📊 For publication figures, use: notebooks/results_visualization.ipynb')